# Sample Problem 0 — Five-Product Pricing

### Load packages

In [1]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory


### Create data

In [2]:
instanceData = {None: {
    'ITEMS': {None: ['A', 'B', 'C', 'D', 'E']},
    'basePrice': dict(zip(['A', 'B', 'C', 'D', 'E'], [20, 25, 30, 40, 50])),
    'initInv': dict(zip(['A', 'B', 'C', 'D', 'E'], [100, 80, 120, 60, 150])),
    'maxPct': {None: 0.70}, 'cap': {None: 60}, 'dispCost': {None: 2},
}}


### Define model

In [3]:
m = AbstractModel()
m.ITEMS = Set()
m.basePrice = Param(m.ITEMS)
m.initInv = Param(m.ITEMS)
m.maxPct = Param()
m.cap = Param()
m.dispCost = Param()

m.MarkPct = Var(m.ITEMS, domain=NonNegativeReals, initialize=0.60)
m.MarkPrice = Var(m.ITEMS, domain=NonNegativeReals)
m.Demand = Var(m.ITEMS, domain=NonNegativeReals)
m.Revenue = Var(m.ITEMS, domain=NonNegativeReals)
m.FinalInv = Var(m.ITEMS, domain=NonNegativeReals)

def MarkPrice_Rule(m, i):
    return m.MarkPrice[i] == m.basePrice[i] * (1 - m.MarkPct[i])
m.Define_MarkPrice = Constraint(m.ITEMS, rule=MarkPrice_Rule)

def Demand_Rule(m, i):
    return m.Demand[i] == m.initInv[i] * m.MarkPct[i]
m.Define_Demand = Constraint(m.ITEMS, rule=Demand_Rule)

def Revenue_Rule(m, i):
    return m.Revenue[i] == m.MarkPrice[i] * m.Demand[i]
m.Define_Revenue = Constraint(m.ITEMS, rule=Revenue_Rule)

def FinalInv_Rule(m, i):
    return m.FinalInv[i] == m.initInv[i] - m.Demand[i]
m.Define_FinalInv = Constraint(m.ITEMS, rule=FinalInv_Rule)

def Max_Discount_Rule(m, i):
    return m.MarkPct[i] <= m.maxPct
m.Max_Discount = Constraint(m.ITEMS, rule=Max_Discount_Rule)

def Inventory_Cap_Rule(m, i):
    return m.FinalInv[i] <= m.cap
m.Inventory_Cap = Constraint(m.ITEMS, rule=Inventory_Cap_Rule)

def Max_Revenue_Rule(m):
    return sum(m.Revenue[i] - m.dispCost * m.FinalInv[i] for i in m.ITEMS)
m.Max_Revenue = Objective(rule=Max_Revenue_Rule, sense=maximize)


### Create instance

In [4]:
instance = m.create_instance(instanceData)


### Solve

In [5]:
solver = SolverFactory('ipopt')
solution = solver.solve(instance)
assert check_optimal_termination(solution), 'Solver did not terminate optimally.'


### Print solution

In [6]:
print(solution['Solver'])

output = pd.DataFrame([
    {'Item': i, **{name: value(getattr(instance, name)[i])
                  for name in ['MarkPct', 'MarkPrice', 'Demand', 'Revenue', 'FinalInv']},
     'NetRevenue': value(instance.Revenue[i] - instance.dispCost * instance.FinalInv[i])}
    for i in instance.ITEMS
])
print(output.round(4).to_string(index=False))
print(f'Total Net Revenue: ${value(instance.Max_Revenue):,.2f}')



- Status: ok
  Message: Ipopt 3.14.20\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.14536333084106445

Item  MarkPct  MarkPrice  Demand  Revenue  FinalInv  NetRevenue
   A   0.5500        9.0    55.0    495.0      45.0       405.0
   B   0.5400       11.5    43.2    496.8      36.8       423.2
   C   0.5333       14.0    64.0    896.0      56.0       784.0
   D   0.5250       19.0    31.5    598.5      28.5       541.5
   E   0.6000       20.0    90.0   1800.0      60.0      1680.0
Total Net Revenue: $3,833.70


### Explanation of results

The shop should discount product A by 55%, B by 54%, C by about 53.33%, D by 52.5%, and E by 60%. These discounts balance the extra sales generated by lower prices against the revenue lost on each pound sold and the cost of disposing of leftover inventory.

The resulting selling prices are $9, $11.50, $14, $19, and $20 per pound for products A through E. The shop sells 55, 43.2, 64, 31.5, and 90 pounds, respectively, leaving 45, 36.8, 56, 28.5, and 60 pounds for disposal.

Product E reaches the maximum allowed leftover inventory of 60 pounds. Its inventory limit forces a larger discount than would otherwise be most profitable. The other four products finish below that limit, and none reaches the maximum permitted discount of 70%.

Total sales revenue is $4,286.30. Disposing of the remaining inventory costs $452.60, leaving total net revenue of $3,833.70. These results assume that demand follows the stated discount relationship and that all predicted demand is served.
